In [ ]:
#install py2neo
!pip install py2neo
!pip install neo4j
!pip install tqdm

In [ ]:
#import class
from py2neo import Graph
import pandas as pd
from neo4j import GraphDatabase
from tqdm import tqdm

#Connect to Neo4j

In [ ]:
URI = "neo4j+s://be0a3f61.databases.neo4j.io"

USERNAME = "be0a3f61"
PASSWORD = "wcEdfl7sSqjEjtNsdGiLIIuWzqZ7m2Q3OwVz47a3N24"

graph = Graph(URI, auth=(USERNAME, PASSWORD))

print("Connected Successfully")

Connected Successfully


#AR-01 What are the Top 10 busiest stops by total passenger  boarding across all transport modes, broken down by zone?

In [ ]:
# Execute query
query = """
MATCH (j:Journey)-[sa:STARTED_AT]->(s:Stop)
RETURN s.zone AS zone,
       s.stop_name AS stop,
       COUNT(j) AS total_boardings
ORDER BY total_boardings DESC
LIMIT 10
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)

#export result
df.to_excel('AR01.xlsx', index=False)

     zone                   stop  total_boardings
0  Zone A    Central Interchange              113
1  Zone F     Kepong LRT Station               72
2  Zone A    Medical Hub Station               70
3  Zone A  Heritage District Hub               65
4  Zone F        Kepong Bus Stop               63
5  Zone F   Jenjarom LRT Station               61
6  Zone B    West Market Station               56
7  Zone A     Hotel District Hub               54
8  Zone D    Ledang BRT Terminal               52
9  Zone A     Kluang LRT Station               50


#AR02 For each day of the week (Monday to Sunday),which two-hour  time window sees the highest number of journey departures on BRT routes?

In [ ]:
# Execute query
query = """
MATCH (j:Journey)-[:USES_ROUTE]->(r:Route)
WHERE r.mode = 'BRT' OR r.route_id STARTS WITH 'BRT'
WITH j,
     substring(j.journey_date, 0, 10) AS date_part,
     // Extract hour (positions 11-12 in format "yyyy-MM-ddTHH:mm:ss")
     substring(j.journey_date, 11, 2) AS hour
WITH date_part, hour,
     COUNT(*) AS departures_per_hour
// Group into two-hour windows
WITH
     date_part,
     CASE
       WHEN toInteger(hour) >= 0 AND toInteger(hour) < 2 THEN '00:00-02:00'
       WHEN toInteger(hour) >= 2 AND toInteger(hour) < 4 THEN '02:00-04:00'
       WHEN toInteger(hour) >= 4 AND toInteger(hour) < 6 THEN '04:00-06:00'
       WHEN toInteger(hour) >= 6 AND toInteger(hour) < 8 THEN '06:00-08:00'
       WHEN toInteger(hour) >= 8 AND toInteger(hour) < 10 THEN '08:00-10:00'
       WHEN toInteger(hour) >= 10 AND toInteger(hour) < 12 THEN '10:00-12:00'
       WHEN toInteger(hour) >= 12 AND toInteger(hour) < 14 THEN '12:00-14:00'
       WHEN toInteger(hour) >= 14 AND toInteger(hour) < 16 THEN '14:00-16:00'
       WHEN toInteger(hour) >= 16 AND toInteger(hour) < 18 THEN '16:00-18:00'
       WHEN toInteger(hour) >= 18 AND toInteger(hour) < 20 THEN '18:00-20:00'
       WHEN toInteger(hour) >= 20 AND toInteger(hour) < 22 THEN '20:00-22:00'
       WHEN toInteger(hour) >= 22 AND toInteger(hour) <= 23 THEN '22:00-24:00'
       ELSE 'Unknown'
     END AS time_window,
     SUM(departures_per_hour) AS total_departures
WITH date_part, time_window, total_departures
WITH time_window,
     SUM(total_departures) AS total_departures
RETURN time_window,
       total_departures
ORDER BY total_departures DESC
LIMIT 1
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)
#export result
df.to_excel('AR02.xlsx', index=False)

   time_window  total_departures
0  18:00-20:00               134


#AR03 Which routes have the highest average delay in minutes, and what percentage of their journey are delayed by more than 15 minutes?

In [ ]:
# Execute query
query = """
MATCH (j:Journey)-[:USES_ROUTE]->(r:Route)
WHERE j.delay_minutes IS NOT NULL
WITH r.route_name AS Route_Name,
     avg(j.delay_minutes) AS avg_delay,
     count(*) AS total_journeys,
     count(CASE WHEN j.delay_minutes > 15 THEN 1 END) AS delayed_over_15
WHERE total_journeys >= 10
RETURN Route_Name,
       round(avg_delay, 2) AS average_delay_minutes,
       round(100.0 * delayed_over_15 / total_journeys, 2) AS pct_delayed_over_15
ORDER BY average_delay_minutes DESC
LIMIT 1
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)
#export result
df.to_excel('AR03.xlsx', index=False)

                 Route_Name  average_delay_minutes  pct_delayed_over_15
0  University Loop – Zone C                  14.96                19.81


#AR04 What is the revenue contribution (total fare collected) per commuter type (Daily, Student, Tourist,etc.) per quarter in 2023?

In [ ]:
# Execute query
query = """
MATCH (p:Passenger)-[:MADE]->(j:Journey)
WITH p.commuter_type AS commuterType,
     datetime(j.journey_date) AS journey_DT,
     j.total_fare_myr AS fare
WHERE journey_DT.year = 2023

RETURN
    commuterType,
    'Q' + toString(toInteger(ceil(journey_DT.month / 3.0))) AS quarter,
    round(SUM(fare),2) AS totalRevenue
ORDER BY quarter, totalRevenue DESC;
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)
#export result
df.to_excel('AR04.xlsx', index=False)

   commuterType quarter  totalRevenue
0         Daily      Q1        337.27
1       Student      Q1        109.05
2    Occasional      Q1         91.88
3        Senior      Q1         54.77
4       Tourist      Q1         35.51
5         Daily      Q2        307.05
6       Student      Q2        115.86
7    Occasional      Q2        104.59
8        Senior      Q2         59.90
9       Tourist      Q2         54.51
10        Daily      Q3        293.42
11      Student      Q3        116.97
12   Occasional      Q3         90.81
13       Senior      Q3         56.46
14      Tourist      Q3         49.89
15        Daily      Q4        270.00
16      Student      Q4        118.19
17   Occasional      Q4        116.87
18      Tourist      Q4         69.73
19       Senior      Q4         44.17


#AR05  How many unique passengers completed at least one multimodal journey(2+transport modes) in a single trip, and what is their average total fare vs single -mode journeys?

In [ ]:
# Execute query
query = """
MATCH (p:Passenger)-[:MADE]->(j:Journey)
WITH p,
     j.total_fare_myr AS fare,
     j.transport_mode AS mode
WHERE j.status <> "Cancelled"

WITH p, fare, mode,
     CASE WHEN mode = "Multimodal"
          THEN "Multimodal"
          ELSE "Single-mode"
     END AS transport_mode

RETURN
    transport_mode,
    count(DISTINCT CASE WHEN transport_mode = "Multimodal" THEN p END) AS multimodalPassengers,
    round(avg(toFloat(fare)),2) AS avgFare,
    round(sum(toFloat(fare)),2) AS totalFare
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)
#export result
df.to_excel('AR05.xlsx', index=False)

  transport_mode  multimodalPassengers  avgFare  totalFare
0    Single-mode                     0     1.59    3004.57
1     Multimodal                   388     3.67    2796.29


#AR06-01 Calculate the total and average CO2 equivalent(in kg) generate per month per transport mode.Which mode has the lowest carbon intensity per km?

In [ ]:
# Execute query
query = """
MATCH (j:Journey)-[:USES_ROUTE]->(r:Route)
WITH
    datetime(j.journey_date) AS dt,
    j.transport_mode AS mode,
    j.carbon_g AS co2_pJ,
    r.distance_km AS km
WHERE j.status <> "Cancelled"

WITH
    dt.year AS year,
    dt.month AS month,
    mode,
    (co2_pJ / 1000) AS co2_kg

RETURN
    year,
    month,
    mode AS transportMode,
    round(sum(co2_kg), 2) AS totalCO2_kg,
    round(avg(co2_kg), 2) AS avgCO2_kg
ORDER BY year, month, transportMode;
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)
#export result
df.to_excel('AR06_01.xlsx', index=False)

     year  month transportMode  totalCO2_kg  avgCO2_kg
0    2022      1           BRT        17.83       1.11
1    2022      1           LRT        10.25       0.51
2    2022      1    Multimodal        28.65       0.99
3    2022      1       Shuttle        13.78       1.53
4    2022      2           BRT        19.72       0.79
..    ...    ...           ...          ...        ...
139  2024     11       Shuttle        10.91       1.56
140  2024     12           BRT        21.99       1.16
141  2024     12           LRT         9.56       0.56
142  2024     12    Multimodal        14.66       0.70
143  2024     12       Shuttle        14.43       1.60

[144 rows x 5 columns]


#AR06-02

In [ ]:
# Execute query
query = """
MATCH (j:Journey)-[:USES_ROUTE]->(r:Route)
WITH
    j.transport_mode AS mode,
    j.carbon_g AS co2_pJ,
    r.distance_km AS km
WHERE j.status <> "Cancelled"

WITH
    mode,
    sum(co2_pJ) / sum(km) AS intensity_pkm

RETURN
    mode AS transportMode,
    round(intensity_pkm, 2) AS co2_per_km
ORDER BY co2_per_km ASC
LIMIT 1
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)

#export result
df.to_excel('AR06_02.xlsx', index=False)

  transportMode  co2_per_km
0           LRT       18.24


#AR07 Which route/stop combination has the most unresolved High or Critical feedback in the last 6 months? List the top 5 along with common tags?

In [ ]:
# Execute query
query = """
MATCH (f:Feedback)-[:ABOUT_ROUTE]->(r:Route)
MATCH (f)-[:ABOUT_STOP]->(s:Stop)
WITH
    f, r, s,
    datetime(f.submitted_at) AS dt
WHERE f.is_resolved = false
  AND f.severity IN ["High", "Critical"]
  AND dt >= datetime("2024-12-30T16:32:08") - duration("P6M")

WITH
    r.route_id AS route,
    s.stop_name AS stop,
    collect(f.tags) AS allTags,
    count(f) AS unresolvedCount

RETURN
    route,
    stop,
    unresolvedCount,
    allTags
ORDER BY unresolvedCount DESC
LIMIT 5;
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)

#export result
df.to_excel('AR07.xlsx', index=False)

    route                   stop  unresolvedCount  \
0  BRT-02       Rawang Micro-Hub                1   
1  BRT-04  Yong Peng LRT Station                1   
2  BRT-05       Serdang Bus Stop                1   
3  BRT-05    Stadium Interchange                1   
4  BRT-02    West Market Station                1   

                                allTags  
0     [[ticketing, app-bug, peak-hour]]  
1               [[praise, bike-damage]]  
2  [[first-mile, last-mile, peak-hour]]  
3            [[overcrowded, peak-hour]]  
4                  [[peak-hour, delay]]  


#AR08 For Gold and Platinum tier passengers, what is the average number of journeys per month,preferred transport mode, and average rating given?

In [ ]:
# Execute query
query = """
MATCH (p:Passenger)-[:MADE]->(j:Journey)
WHERE p.loyalty_tier IN ['Gold', 'Platinum']
WITH p,
     LEFT(j.journey_date, 7) AS year_month,
     j.transport_mode AS mode,
     CASE
       WHEN j.rating IS NULL OR toString(j.rating) = 'NaN' THEN 0.0
       ELSE j.rating
     END AS rating
// Per passenger per month
WITH p, year_month, mode,
     COUNT(*) AS monthly_journeys,
     AVG(rating) AS monthly_rating
// Per passenger
WITH p,
     AVG(monthly_journeys) AS avg_monthly,
     AVG(monthly_rating) AS avg_rating,
     COLLECT(DISTINCT mode) AS modes_used
// Per passenger - find preferred mode
WITH p,
     avg_monthly,
     avg_rating,
     [m IN modes_used WHERE m IS NOT NULL AND m <> ""][0] AS preferred_mode
// Group by loyalty_tier
WITH p.loyalty_tier AS loyalty_tier,
     ROUND(AVG(avg_monthly), 2) AS avg_journeys_per_month,
     COLLECT(preferred_mode) AS all_modes,
     ROUND(AVG(avg_rating), 2) AS avg_rating_given
WITH loyalty_tier,
     avg_journeys_per_month,
     avg_rating_given,
     all_modes,
     // Find most common preferred mode
     REDUCE(best = {mode: "", count: 0}, m IN all_modes |
       CASE
         WHEN m IS NOT NULL AND m <> "" AND SIZE([x IN all_modes WHERE x = m]) > best.count
         THEN {mode: m, count: SIZE([x IN all_modes WHERE x = m])}
         ELSE best
       END
     ) AS top_mode
RETURN loyalty_tier,
       avg_journeys_per_month,
       top_mode.mode AS preferred_transport_mode,
       avg_rating_given
ORDER BY loyalty_tier
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)

#export result
df.to_excel('AR08.xlsx', index=False)

  loyalty_tier  avg_journeys_per_month preferred_transport_mode  \
0         Gold                    1.00               Multimodal   
1     Platinum                    1.02               Multimodal   

   avg_rating_given  
0              1.67  
1              1.97  


#AR09 Using the interchange connection data in the database, find all stops reachable from Central Interchange (STOP-001) within 2 transfer hops. For each reachable stop, return its name,zone, and the routes available at that stop.

In [ ]:
# Execute query
query = """
MATCH (start:Stop {stop_id: "STOP-001"})
CALL {
  WITH start
  // 1 hop
  MATCH (start)<-[:SERVES]-(r:Route)-[:SERVES]->(stop1:Stop)
  WHERE stop1 <> start
  RETURN DISTINCT stop1 AS stop, r AS route, 1 AS hops

  UNION

  // 2 hops
  MATCH (start)<-[:SERVES]-(r1:Route)-[:SERVES]->(mid:Stop)<-[:SERVES]-(r2:Route)-[:SERVEST]->(stop2:Stop)
  WHERE mid <> start
    AND stop2 <> start
    AND stop2 <> mid
  RETURN DISTINCT stop2 AS stop, r2 AS route, 2 AS hops
}
WITH stop, hops, COLLECT(DISTINCT route) AS routes
RETURN
       stop.stop_name AS stop_name,
       stop.zone AS zone,
       [r IN routes | r.route_id] AS route_ids,
       [r IN routes | r.route_name] AS route_names
ORDER BY hops, stop.stop_name
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)

#export result
df.to_excel('AR09.xlsx', index=False)

                    stop_name    zone              route_ids  \
0        Airport Terminal Hub  Zone F                [LRT-A]   
1      Ampang Pecah Micro-Hub  Zone C               [BRT-11]   
2              Bahau Bus Stop  Zone F        [BRT-06, LRT-A]   
3             Bangi Micro-Hub  Zone E               [BRT-01]   
4        Buloh Kasap Bus Stop  Zone F                [LRT-A]   
5          Cheras LRT Station  Zone F                [LRT-B]   
6    Desa Tebrau BRT Terminal  Zone D                [LRT-C]   
7        Desa Tebrau Bus Stop  Zone B        [BRT-06, LRT-A]   
8   East Terminal Interchange  Zone E                [LRT-B]   
9     Gemas North LRT Station  Zone F               [BRT-01]   
10       Genuang BRT Terminal  Zone F               [BRT-01]   
11        Gombak BRT Terminal  Zone C               [BRT-06]   
12      Heritage District Hub  Zone A                [LRT-C]   
13         Hotel District Hub  Zone A        [BRT-01, LRT-C]   
14    Hulu Langat LRT Station  Zone E   

#AR10 Identify passengers who have more than 20% of their journeys with status ‘Cancelled’ or ‘Incomplete’ in 2024, along with their total journey count and home zone.

In [ ]:
# Execute query
query = """
MATCH (p:Passenger)-[:MADE]->(j:Journey)
WHERE j.journey_date STARTS WITH '2024'
WITH p,
     COUNT(j) AS total_journeys,
     SUM(CASE WHEN j.status IN ['Cancelled', 'Incomplete'] THEN 1 ELSE 0 END) AS bad_journeys
WHERE total_journeys > 0  // Avoid division by zero
  AND (bad_journeys * 1.0 / total_journeys) > 0.20
RETURN p.passenger_id AS passenger_id,
       p.full_name AS full_name,
       p.home_zone AS home_zone,
       total_journeys,
       bad_journeys AS cancelled_incomplete_count,
      ROUND((bad_journeys * 100.0 / total_journeys), 2) AS bad_percentage
ORDER BY bad_percentage DESC
"""

# Run query and get results
results = graph.run(query).data()

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)
#export result
df.to_excel('AR10.xlsx', index=False)

                             passenger_id             full_name home_zone  \
0    fcb1a281-2d53-4810-bc01-c46c54ac13ce     Ms Hilary Walters    Zone A   
1    702805af-5806-41f1-b820-242631341ab8         Joel Stephens    Zone B   
2    5fc57ff4-d7a9-48ea-af7c-db39283edeec  Terry Smith-Harrison    Zone D   
3    f20ba0c9-2a86-4922-b492-7b3536c21bc3          Lydia Bishop    Zone D   
4    970e287a-0fd7-40f2-8f90-8490006e0070         Patrick Jones    Zone F   
..                                    ...                   ...       ...   
234  98ab182c-3b1d-4181-9cc3-c1f59129bc75  Miss Alice Tomlinson    Zone B   
235  b1f94a09-7408-4646-93a7-1719bcdf8302       Benjamin Peters    Zone A   
236  d94296b6-7e4b-4f10-b45f-780efbe5403b      Dr Mathew Fowler    Zone A   
237  89b4db80-8a32-484a-a401-461af5b92b3c         Terence Davis    Zone E   
238  29769b0f-41e0-4faf-92ca-9cedc5268f0f       Stephanie Lloyd    Zone A   

     total_journeys  cancelled_incomplete_count  bad_percentage  
0        

#export file

In [ ]:

driver = GraphDatabase.driver(URI,
                               auth=(USERNAME,PASSWORD))


with driver.session() as session:
    result = session.run("""
    CALL apoc.export.json.all(null, {stream: true})
    """)

    record = result.single()

    if record:
        # APOC sometimes returns different formats depending on version
        data = record[0] or record.get("data")
    else:
        data = None

if data:
    with open("export.json", "w") as f:
        f.write(data)
    print("Export saved successfully!")
else:
    print("Export failed or returned empty result")

Export saved successfully!
